# CredibleX Growth Intelligence Engine
## 02 · Opportunity Engine (Python + SQL)

This notebook implements the four core analytical modules:

- **A. SME Financing Opportunity Score** (0–100)
- **B. Financing Product Recommendation**
- **C. Partner Opportunity Score** (0–100)
- **D. Early Warning Classification**

These are illustrative **commercial-opportunity** heuristics, not credit-risk
models, and are not CredibleX's actual underwriting methodology.

SQL versions of the same logic are provided in `../sql/analysis.sql` for
querying the datasets from SQL Server / any SQL engine directly.

In [1]:
import pandas as pd
import numpy as np
import sqlite3

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

sme = pd.read_csv('../data/sme_data_clean.csv')
partner = pd.read_csv('../data/partner_data_clean.csv')
print(sme.shape, partner.shape)

(1000, 13) (50, 9)


## A. SME Financing Opportunity Score

The score blends five normalised (min-max scaled) signals:

| Signal | Weight | Rationale |
|---|---|---|
| Revenue growth | 25% | Growing SMEs have expanding financing needs |
| Financing requirement (relative to revenue) | 25% | Direct proxy for financing demand |
| Working-capital pressure (receivable days & low cash buffer) | 20% | Cash-flow timing gaps create financing need |
| Transaction volume | 15% | Higher activity = more embeddable financing touchpoints |
| Revenue stability (inverse of volatility) | 15% | Stable revenue supports serviceable financing |


In [2]:
def minmax(s):
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi != lo else pd.Series(np.zeros(len(s)), index=s.index)

growth_n = minmax(sme['Revenue_Growth'].clip(lower=0))
financing_n = minmax(sme['Financing_Requirement'] / sme['Monthly_Revenue'])
wc_pressure_n = minmax(0.6*minmax(sme['Receivable_Days']) + 0.4*(1 - minmax(sme['Cash_Buffer'])))
txn_n = minmax(np.log1p(sme['Transaction_Volume']))
stability_n = 1 - minmax(sme['Revenue_Volatility'])

sme['Opportunity_Score'] = (
    0.25*growth_n + 0.25*financing_n + 0.20*wc_pressure_n + 0.15*txn_n + 0.15*stability_n
) * 100
sme['Opportunity_Score'] = sme['Opportunity_Score'].round(1)

q65, q40, q15 = sme['Opportunity_Score'].quantile([0.65, 0.40, 0.15])
def bucket(s):
    if s >= q65 and s >= 60: return 'High Opportunity'
    elif s >= q40: return 'Emerging Opportunity'
    elif s >= q15: return 'Moderate Opportunity'
    else: return 'Low Opportunity'

sme['Opportunity_Segment'] = sme['Opportunity_Score'].apply(bucket)
sme['Opportunity_Segment'].value_counts()

Opportunity_Segment
Emerging Opportunity    542
Moderate Opportunity    252
Low Opportunity         147
High Opportunity         59
Name: count, dtype: int64

In [3]:
sme.nlargest(10, 'Opportunity_Score')[
    ['SME_ID','Industry','Emirate','Monthly_Revenue','Revenue_Growth',
     'Receivable_Days','Opportunity_Score','Opportunity_Segment']
]

,SME_ID,Industry,Emirate,Monthly_Revenue,Revenue_Growth,Receivable_Days,Opportunity_Score,Opportunity_Segment
45,SME0046,Professional Services,Dubai,424660,0.448,133,77.8,High Opportunity
129,SME0130,Construction,Umm Al Quwain,349780,0.589,42,75.2,High Opportunity
745,SME0746,F&B,Abu Dhabi,43785,0.645,59,73.8,High Opportunity
903,SME0904,F&B,Dubai,108095,0.619,137,73.0,High Opportunity
805,SME0806,Retail,Sharjah,190205,0.324,118,72.3,High Opportunity
394,SME0395,Retail,Ajman,79678,0.503,58,71.7,High Opportunity
284,SME0285,Construction,Fujairah,263032,0.456,55,71.0,High Opportunity
742,SME0743,Trading,Dubai,219466,0.271,148,69.9,High Opportunity
755,SME0756,E-commerce,Abu Dhabi,92655,0.650,38,69.6,High Opportunity
593,SME0594,Healthcare,Dubai,70148,0.628,46,69.4,High Opportunity


## B. Financing Product Recommendation

Simple rule-based matcher across three products:
- **Receivables Financing** — when receivable days are high relative to payable days
- **Revenue-Based Financing** — when growth is strong and revenue is stable
- **Payable Financing** — when payable days are high (extend supplier terms)

In [4]:
def recommend_product(row):
    rd, pd_, growth, vol = row['Receivable_Days'], row['Payable_Days'], row['Revenue_Growth'], row['Revenue_Volatility']
    if rd >= 45 and rd >= pd_:
        return 'Receivables Financing', (
            f"Receivable days of {rd} exceed payable days of {pd_}, suggesting delayed "
            "customer payments are creating working-capital pressure that invoice/receivables "
            "financing could bridge.")
    elif growth >= 0.15 and vol <= 0.35:
        return 'Revenue-Based Financing', (
            f"Revenue growth of {growth*100:.1f}% combined with relatively stable revenue "
            "indicates the business could support financing repaid as a share of future revenue.")
    elif pd_ >= 30:
        return 'Payable Financing', (
            f"Payable days of {pd_} suggest the SME could benefit from extending supplier "
            "payment terms through payable financing to smooth cash flow.")
    else:
        return 'Revenue-Based Financing', (
            "No single working-capital driver dominates; revenue-based financing offers the "
            "most flexible fit given the business's current profile.")

results = sme.apply(recommend_product, axis=1, result_type='expand')
sme['Recommended_Product'] = results[0]
sme['Recommendation_Reason'] = results[1]
sme['Recommended_Product'].value_counts()

Recommended_Product
Revenue-Based Financing    419
Receivables Financing      378
Payable Financing          203
Name: count, dtype: int64

In [5]:
sme.groupby('Recommended_Product')['Opportunity_Score'].agg(['mean','count']).round(1)

,mean,count
Recommended_Product,,
Payable Financing,34.3,203
Receivables Financing,45.7,378
Revenue-Based Financing,39.8,419


## C. Partner Opportunity Score

Blends reach, activity, financing relevance, digital maturity, and growth —
with a penalty for integration complexity.

In [6]:
reach_n = minmax(np.log1p(partner['SME_Reach']))
txn_n_p = minmax(np.log1p(partner['Transaction_Volume']))
relevance_n = minmax(partner['Financing_Relevance'])
digital_n = minmax(partner['Digital_Maturity'])
growth_n_p = minmax(partner['Growth_Rate'])
complexity_penalty = partner['Integration_Complexity'].map({'Low':0.0,'Medium':0.08,'High':0.18})

score = (0.28*reach_n + 0.20*txn_n_p + 0.27*relevance_n + 0.10*digital_n + 0.15*growth_n_p)*100 - complexity_penalty*100
partner['Partner_Score'] = score.clip(lower=0, upper=100).round(1)

q70, q35 = partner['Partner_Score'].quantile([0.70, 0.35])
def pbucket(s):
    if s >= q70: return 'Priority'
    elif s >= q35: return 'Emerging'
    else: return 'Monitor'

partner['Partner_Segment'] = partner['Partner_Score'].apply(pbucket)
partner['Partner_Segment'].value_counts()

Partner_Segment
Monitor     18
Emerging    17
Priority    15
Name: count, dtype: int64

In [7]:
partner.nlargest(10, 'Partner_Score')[
    ['Partner_ID','Partner_Type','SME_Reach','Financing_Relevance','Partner_Score','Partner_Segment']
]

,Partner_ID,Partner_Type,SME_Reach,Financing_Relevance,Partner_Score,Partner_Segment
43,PTN044,Supplier Network,5919,94.5,81.4,Priority
42,PTN043,Supplier Network,40914,38.8,79.5,Priority
49,PTN050,Free Zone Authority,6094,63.5,63.5,Priority
23,PTN024,ERP / Invoicing Platform,3135,24.7,58.9,Priority
40,PTN041,E-commerce Platform,6533,86.0,58.9,Priority
38,PTN039,B2B Marketplace,3354,24.5,58.2,Priority
39,PTN040,Logistics Aggregator,8930,53.4,58.0,Priority
44,PTN045,B2B Marketplace,2427,72.1,56.8,Priority
32,PTN033,Free Zone Authority,519,82.0,56.7,Priority
28,PTN029,Logistics Aggregator,7531,43.9,55.9,Priority


## D. Early Warning Classification

Flags SMEs on four risk-adjacent signals; 3+ flags = Attention Required,
1–2 = Monitor, 0 = Stable.

In [8]:
flags = pd.DataFrame(index=sme.index)
flags['declining_revenue'] = sme['Revenue_Growth'] < -0.05
flags['high_receivables'] = sme['Receivable_Days'] > sme['Receivable_Days'].quantile(0.75)
flags['low_transactions'] = sme['Transaction_Volume'] < sme['Transaction_Volume'].quantile(0.25)
flags['low_cash_buffer'] = sme['Cash_Buffer'] < sme['Cash_Buffer'].quantile(0.25)
flags['poor_repayment'] = sme['Repayment_Behaviour'].isin(['Poor','Fair'])

sme['Warning_Flags'] = flags.sum(axis=1)
def wbucket(n):
    if n >= 3: return 'Attention Required'
    elif n >= 1: return 'Monitor'
    else: return 'Stable'
sme['Early_Warning_Status'] = sme['Warning_Flags'].apply(wbucket)
sme['Early_Warning_Status'].value_counts()

Early_Warning_Status
Monitor               694
Stable                189
Attention Required    117
Name: count, dtype: int64

### Cross-tab: Opportunity Segment vs Early Warning Status
High-opportunity SMEs are not automatically risk-free — this table shows where the two views diverge.

In [9]:
pd.crosstab(sme['Opportunity_Segment'], sme['Early_Warning_Status'])

Early_Warning_Status,Attention Required,Monitor,Stable
Opportunity_Segment,,,
Emerging Opportunity,65,365,112
High Opportunity,6,43,10
Low Opportunity,19,108,20
Moderate Opportunity,27,178,47


## Querying with SQL
The same datasets can be queried directly with SQL (see `../sql/analysis.sql` for the full set). Demonstrated here in-notebook using an in-memory SQLite database.

In [10]:
conn = sqlite3.connect(':memory:')
sme.to_sql('sme', conn, index=False, if_exists='replace')
partner.to_sql('partner', conn, index=False, if_exists='replace')

query = '''
SELECT Industry,
       COUNT(*) AS SME_Count,
       ROUND(AVG(Opportunity_Score), 1) AS Avg_Opportunity_Score,
       SUM(CASE WHEN Opportunity_Segment = 'High Opportunity' THEN 1 ELSE 0 END) AS High_Opportunity_Count
FROM sme
GROUP BY Industry
ORDER BY Avg_Opportunity_Score DESC;
'''
pd.read_sql(query, conn)

,Industry,SME_Count,Avg_Opportunity_Score,High_Opportunity_Count
0,Construction,74,44.2,6
1,Healthcare,73,41.9,4
2,Trading,95,41.7,8
3,Logistics,94,41.5,4
4,Manufacturing,83,41.3,6
5,Retail,176,40.5,5
6,E-commerce,121,40.5,6
7,Hospitality,56,40.0,4
8,F&B,122,39.9,10
9,Professional Services,106,39.4,6


## Save scored datasets
These enriched datasets feed the AI layer (`ai_insights.ipynb`) and the
HTML dashboard.

In [11]:
sme.to_csv('../data/sme_scored.csv', index=False)
partner.to_csv('../data/partner_scored.csv', index=False)
print("Saved: data/sme_scored.csv, data/partner_scored.csv")
print(f"High-Opportunity SMEs: {(sme['Opportunity_Segment']=='High Opportunity').sum()}")
print(f"Priority Partners: {(partner['Partner_Segment']=='Priority').sum()}")

Saved: data/sme_scored.csv, data/partner_scored.csv
High-Opportunity SMEs: 59
Priority Partners: 15


**Next:** `ai_insights.ipynb` — turns these calculated results into AI-generated business briefs and an executive summary.